# Feature-Based EDA: AI Music Detection & Pairwise Similarity

This notebook analyzes precomputed features across all four datasets:
- **SONICS**: Lyric pairs (real vs AI), sibling pairs (AI vs AI), extra fakes/reals
- **FakeMusicCaps**: Multiple AI generators (MusicGen, etc.)
- **SMP**: Real plagiarism/cover pairs
- **Echoes**: TTA/ATA generated tracks with FMA originals

Features analyzed:
- **Classical (432-d)**: MFCCs, mel-spectrogram, chroma, tonnetz, spectral contrast, temporal
- **AI Detection (22-d)**: Phase continuity, spectral AI indicators, Fourier artifact summary
- **Fakeprint (897-d)**: Afchar et al. ISMIR 2025 — vocoder stride artifacts
- **Embeddings (1536-d)**: MERT (1024-d) + CLAP (512-d)

**Prerequisite**: Run `python precompute_all.py` first to generate `feature_cache/`.

In [90]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import cosine
from scipy.stats import mannwhitneyu, kruskal
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 120

CACHE_DIR = Path('../feature_cache_merged')
FIG_DIR = Path('feature_figs')
FIG_DIR.mkdir(exist_ok=True)

def save_plot(name):
    plt.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight', dpi=150)
    plt.close()

# Classical feature group names (for interpretation)
CLASSICAL_GROUPS = {
    'MFCC': (0, 120),
    'Mel-spectrogram': (120, 376),
    'Chroma': (376, 400),
    'Tonnetz': (400, 412),
    'Spectral Contrast': (412, 426),
    'Temporal': (426, 432),
}

AI_DETECTION_GROUPS = {
    'Phase Continuity': (0, 7),
    'Spectral AI Indicators': (7, 12),
    'Fourier Artifact Summary': (12, 22),
}

print(f'Cache dir: {CACHE_DIR}')
print(f'Figure dir: {FIG_DIR}')

Cache dir: ../feature_cache_merged
Figure dir: feature_figs


In [91]:
# === Load feature metadata and features ===
metadata_path = CACHE_DIR / 'feature_metadata.csv'
if not metadata_path.exists():
    raise FileNotFoundError(f"Could not find metadata at {metadata_path}")

meta = pd.read_csv(metadata_path, low_memory=False)
print(f'Total files in metadata: {len(meta)}')

# Load features from cache
def load_features(row, cache_dir):
    """Load features from .npz cache file, resolving paths correctly."""
    # Resolve the path by taking just the filename and joining with CACHE_DIR
    filename = Path(row['cache_path']).name
    actual_path = cache_dir / filename
    
    if not actual_path.exists():
        return None
    try:
        # Using 'with' ensures the file is closed after reading
        with np.load(actual_path, allow_pickle=True) as data:
            return {
                'classical': data['classical'] if 'classical' in data else None,
                'ai_detection': data['ai_detection'] if 'ai_detection' in data else None,
                'fakeprint': data['fakeprint'] if 'fakeprint' in data else None,
                'mert': data['mert'] if 'mert' in data else None,
                'clap': data['clap'] if 'clap' in data else None,
            }
    except Exception as e:
        print(f"Error loading {actual_path}: {e}")
        return None

# Load all features into arrays
print('Loading features from cache...')
classical_list, ai_det_list, fakeprint_list = [], [], []
mert_list, clap_list = [], []
valid_idx = []

for i, row in meta.iterrows():
    feats = load_features(row, CACHE_DIR)
    
    # Critical Check: Ensure all mandatory features are present
    if feats is None or feats['classical'] is None or feats['ai_detection'] is None:
        continue
        
    classical_list.append(feats['classical'])
    ai_det_list.append(feats['ai_detection'])
    fakeprint_list.append(feats['fakeprint'])
    
    # Handle optional embeddings
    mert_list.append(feats['mert'] if feats['mert'] is not None else np.zeros(1024, dtype=np.float32))
    clap_list.append(feats['clap'] if feats['clap'] is not None else np.zeros(512, dtype=np.float32))
    
    valid_idx.append(i)

# Check if we loaded anything before proceeding
if not valid_idx:
    raise ValueError(f"No valid features found in {CACHE_DIR}. Check your paths.")

X_classical = np.array(classical_list)
X_ai_det = np.array(ai_det_list)
X_fakeprint = np.array(fakeprint_list)
X_mert = np.array(mert_list)
X_clap = np.array(clap_list)

df = meta.loc[valid_idx].reset_index(drop=True)
df['is_ai'] = df['is_ai'].astype(bool)

# Check if embeddings are actually computed (not all zeros)
has_mert = np.any(X_mert != 0)
has_clap = np.any(X_clap != 0)

print(f'Loaded features for {len(df)} files')
print(f'Classical: {X_classical.shape}, AI Detection: {X_ai_det.shape}, Fakeprint: {X_fakeprint.shape}')

Total files in metadata: 10178
Loading features from cache...
Loaded features for 10178 files
Classical: (10178, 432), AI Detection: (10178, 22), Fakeprint: (10178, 897)


## 1. Real vs AI: Feature Distributions Across All Datasets
How do classical and AI-detection features differ between real and AI-generated tracks?

In [92]:
# === 1a. AI Detection Features: Real vs Fake (all datasets combined) ===

# Verify dimensions match your ai_detection.py constants (22 dims)
if X_ai_det.ndim != 2 or X_ai_det.shape[1] != 22:
    raise ValueError(f"Expected 22 features for AI detection, got {X_ai_det.shape}")

ai_feat_names = [
    'PCI', 'Phase Dev Mean', 'Phase Dev Std',
    'IF Stability Mean', 'IF Stability Std',
    'GD Dev Mean', 'GD Dev Std',
    'HNR', 'Flatness Mean', 'Flatness Std', 'Rolloff Ratio', 'SSM Novelty',
    'PTB 2x', 'PTB 4x', 'PTB 8x',
    'PeakCnt 2x', 'PeakCnt 4x', 'PeakCnt 8x',
    'Peak Regularity', 'Max PTB', 'Spectral Periodicity', 'Artifact Energy'
]

df_ai = pd.DataFrame(X_ai_det, columns=ai_feat_names)
df_ai['is_ai'] = df['is_ai'].values
df_ai['dataset'] = df['dataset'].values

# Box plots for each AI detection feature
# We use 4x6 grid to accommodate 22 features
fig, axes = plt.subplots(4, 6, figsize=(24, 16))
axes = axes.ravel()

for i, feat in enumerate(ai_feat_names):
    if i >= len(axes): break
    sns.boxplot(data=df_ai, x='is_ai', y=feat, hue='is_ai', legend=False, ax=axes[i])
    axes[i].set_title(feat, fontsize=9)
    axes[i].set_xlabel('')
    axes[i].set_xticklabels(['Real', 'AI'])

# Hide empty subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('AI Detection Features: Real vs AI (All Datasets)', fontsize=14, y=1.01)
plt.tight_layout()
save_plot('01_ai_detection_real_vs_fake_all')

# Statistical tests
print('Mann-Whitney U tests (Real vs AI):')
for feat in ai_feat_names:
    real_vals = df_ai[~df_ai['is_ai']][feat].dropna()
    ai_vals = df_ai[df_ai['is_ai']][feat].dropna()
    if len(real_vals) > 5 and len(ai_vals) > 5:
        stat, p = mannwhitneyu(real_vals, ai_vals, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {feat:25s}: p={p:.2e} {sig}  (real_median={real_vals.median():.4f}, ai_median={ai_vals.median():.4f})')

Mann-Whitney U tests (Real vs AI):
  PCI                      : p=1.81e-44 ***  (real_median=1.4892, ai_median=1.4941)
  Phase Dev Mean           : p=8.26e-33 ***  (real_median=1.8119, ai_median=1.8107)
  Phase Dev Std            : p=0.00e+00 ***  (real_median=0.0148, ai_median=0.0433)
  IF Stability Mean        : p=8.15e-87 ***  (real_median=2.7698, ai_median=2.7568)
  IF Stability Std         : p=3.76e-08 ***  (real_median=1.7164, ai_median=1.7207)
  GD Dev Mean              : p=2.53e-01 ns  (real_median=2.2824, ai_median=2.2869)
  GD Dev Std               : p=8.64e-17 ***  (real_median=0.2430, ai_median=0.2108)
  HNR                      : p=9.27e-126 ***  (real_median=-0.3335, ai_median=1.3896)
  Flatness Mean            : p=4.48e-187 ***  (real_median=0.0345, ai_median=0.0138)
  Flatness Std             : p=2.03e-223 ***  (real_median=0.0564, ai_median=0.0188)
  Rolloff Ratio            : p=5.13e-02 ns  (real_median=0.6784, ai_median=0.6849)
  SSM Novelty              : p=2.58e-16

In [93]:
# === 1b. AI Detection Features per Dataset (Real vs Fake) ===
datasets_with_both = []
for ds in df['dataset'].unique():
    sub = df[df['dataset'] == ds]
    if sub['is_ai'].any() and (~sub['is_ai']).any():
        datasets_with_both.append(ds)

# Key AI features to compare per dataset
key_ai_feats = ['PCI', 'HNR', 'Spectral Periodicity', 'Artifact Energy', 'Peak Regularity', 'Max PTB']

for feat in key_ai_feats:
    fig, axes = plt.subplots(1, len(datasets_with_both), figsize=(5*len(datasets_with_both), 4))
    if len(datasets_with_both) == 1:
        axes = [axes]
    for ax, ds in zip(axes, datasets_with_both):
        mask = df['dataset'] == ds
        sub = df_ai[mask]
        sns.boxplot(data=sub, x='is_ai', y=feat, hue='is_ai', legend=False, ax=ax)
        ax.set_title(f'{ds}', fontsize=10)
        ax.set_xticklabels(['Real', 'AI'])
        ax.set_xlabel('')
    plt.suptitle(f'{feat}: Real vs AI per Dataset', fontsize=12)
    plt.tight_layout()
    save_plot(f'01b_{feat.lower().replace(" ", "_")}_per_dataset')

print('Per-dataset AI detection comparison complete.')

Per-dataset AI detection comparison complete.


In [94]:
# === 1c. Classical Features: Real vs AI ===
# Summarize each classical group as mean across dims
classical_summary = {}
for group_name, (start, end) in CLASSICAL_GROUPS.items():
    classical_summary[f'{group_name}_mean'] = np.mean(X_classical[:, start:end], axis=1)
    classical_summary[f'{group_name}_std'] = np.std(X_classical[:, start:end], axis=1)

df_cls = pd.DataFrame(classical_summary)
df_cls['is_ai'] = df['is_ai'].values
df_cls['dataset'] = df['dataset'].values
df_cls['algorithm'] = df['algorithm'].values

# Classical feature group means: real vs fake
mean_cols = [c for c in df_cls.columns if c.endswith('_mean')]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()
for i, col in enumerate(mean_cols):
    if i >= len(axes): break
    sns.boxplot(data=df_cls, x='is_ai', y=col, hue='is_ai', legend=False, ax=axes[i])
    axes[i].set_title(col.replace('_mean', ''), fontsize=10)
    axes[i].set_xticklabels(['Real', 'AI'])
    axes[i].set_xlabel('')
plt.suptitle('Classical Feature Groups (Mean): Real vs AI', fontsize=13)
plt.tight_layout()
save_plot('01c_classical_real_vs_fake')

# Statistical tests for classical features
print('Classical features Mann-Whitney U (Real vs AI):')
for col in mean_cols:
    r = df_cls[~df_cls['is_ai']][col].dropna()
    a = df_cls[df_cls['is_ai']][col].dropna()
    if len(r) > 5 and len(a) > 5:
        stat, p = mannwhitneyu(r, a, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {col:30s}: p={p:.2e} {sig}')

Classical features Mann-Whitney U (Real vs AI):
  MFCC_mean                     : p=1.42e-84 ***
  Mel-spectrogram_mean          : p=1.08e-57 ***
  Chroma_mean                   : p=9.98e-19 ***
  Tonnetz_mean                  : p=1.67e-03 **
  Spectral Contrast_mean        : p=3.10e-169 ***
  Temporal_mean                 : p=1.45e-01 ns


## 2. What AI Songs Have in Common (Across All Datasets)

In [95]:
# === 2a. Fakeprint Analysis: AI common patterns ===
mask_ai = df['is_ai'].values
mask_real = ~mask_ai

# Average fakeprint: AI vs Real
fp_ai_mean = np.mean(X_fakeprint[mask_ai], axis=0)
fp_real_mean = np.mean(X_fakeprint[mask_real], axis=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
freq_axis = np.linspace(1000, 8000, len(fp_ai_mean))  # 1-8 kHz

axes[0].plot(freq_axis, fp_ai_mean, 'r-', alpha=0.8, label='AI (mean)')
axes[0].plot(freq_axis, fp_real_mean, 'b-', alpha=0.8, label='Real (mean)')
axes[0].fill_between(freq_axis, fp_ai_mean, fp_real_mean, alpha=0.2, color='gray')
axes[0].set_title('Average Fakeprint: AI vs Real (All Datasets)')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Peak Height (dB above envelope)')
axes[0].legend()

# Difference
diff = fp_ai_mean - fp_real_mean
axes[1].bar(freq_axis, diff, width=8, color=np.where(diff > 0, 'red', 'blue'), alpha=0.6)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Fakeprint Difference (AI - Real)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Difference (dB)')
plt.tight_layout()
save_plot('02a_fakeprint_ai_vs_real')

# Fakeprint energy statistics
fp_energy = np.sum(X_fakeprint, axis=1)
fp_max = np.max(X_fakeprint, axis=1)
fp_n_peaks = np.sum(X_fakeprint > (np.mean(X_fakeprint, axis=1, keepdims=True) +
                                    2 * np.std(X_fakeprint, axis=1, keepdims=True)), axis=1)

print(f'Fakeprint total energy — AI: {fp_energy[mask_ai].mean():.1f} +/- {fp_energy[mask_ai].std():.1f}')
print(f'                        Real: {fp_energy[mask_real].mean():.1f} +/- {fp_energy[mask_real].std():.1f}')
print(f'Fakeprint max peak    — AI: {fp_max[mask_ai].mean():.2f}, Real: {fp_max[mask_real].mean():.2f}')
print(f'Fakeprint peak count  — AI: {fp_n_peaks[mask_ai].mean():.1f}, Real: {fp_n_peaks[mask_real].mean():.1f}')

Fakeprint total energy — AI: 3696.4 +/- 881.7
                        Real: 2686.4 +/- 1079.2
Fakeprint max peak    — AI: 44.11, Real: 22.61
Fakeprint peak count  — AI: 37.6, Real: 40.9


In [96]:
# === 2b. PCA/t-SNE of AI detection features ===
# Combine ai_detection + fakeprint summary stats for visualization
fp_stats = np.column_stack([fp_energy, fp_max, fp_n_peaks])
X_combined_ai = np.hstack([X_ai_det, fp_stats])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined_ai)
X_scaled = np.nan_to_num(X_scaled)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Color by real/AI
for label, color, name in [(True, 'red', 'AI'), (False, 'blue', 'Real')]:
    mask = df['is_ai'] == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, alpha=0.3, s=10, label=name)
axes[0].set_title(f'PCA of AI Detection Features (var: {pca.explained_variance_ratio_.sum():.1%})')
axes[0].legend()
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Color by dataset
for ds in df['dataset'].unique():
    mask = df['dataset'] == ds
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.3, s=10, label=ds)
axes[1].set_title('PCA of AI Detection Features (by Dataset)')
axes[1].legend()
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.tight_layout()
save_plot('02b_ai_features_pca')

# t-SNE (subsample if large)
n_tsne = min(3000, len(X_scaled))
idx_tsne = np.random.choice(len(X_scaled), n_tsne, replace=False)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_scaled[idx_tsne])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
df_sub = df.iloc[idx_tsne]

for label, color, name in [(True, 'red', 'AI'), (False, 'blue', 'Real')]:
    mask = df_sub['is_ai'].values == label
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=color, alpha=0.4, s=10, label=name)
axes[0].set_title('t-SNE: AI Detection Features (Real vs AI)')
axes[0].legend()

for ds in df_sub['dataset'].unique():
    mask = df_sub['dataset'].values == ds
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], alpha=0.4, s=10, label=ds)
axes[1].set_title('t-SNE: AI Detection Features (by Dataset)')
axes[1].legend()
plt.tight_layout()
save_plot('02b_ai_features_tsne')

print('AI features visualization complete.')

AI features visualization complete.


## 3. AI Songs Per Algorithm: Common Patterns

In [97]:
# === 3a. Fakeprint per algorithm ===
ai_mask = df['is_ai'].values
algorithms = df.loc[ai_mask, 'algorithm'].unique()
algorithms = [a for a in algorithms if pd.notna(a) and a != 'unknown' and a != 'real']
print(f'Algorithms found: {algorithms}')

# Average fakeprint per algorithm
n_algos = len(algorithms)
n_cols = min(3, n_algos)
n_rows = (n_algos + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 4*n_rows))
if n_algos == 1:
    axes = np.array([axes])
axes = np.array(axes).ravel()

for i, algo in enumerate(sorted(algorithms)):
    if i >= len(axes): break
    mask = (df['algorithm'] == algo).values & ai_mask
    n = mask.sum()
    if n == 0: continue
    fp_mean = np.mean(X_fakeprint[mask], axis=0)
    fp_std = np.std(X_fakeprint[mask], axis=0)
    axes[i].plot(freq_axis, fp_mean, 'r-', linewidth=0.8)
    axes[i].fill_between(freq_axis, fp_mean - fp_std, fp_mean + fp_std, alpha=0.2, color='red')
    axes[i].plot(freq_axis, fp_real_mean, 'b--', alpha=0.5, linewidth=0.5, label='Real baseline')
    axes[i].set_title(f'{algo} (n={n})', fontsize=10)
    axes[i].set_xlabel('Freq (Hz)')
    axes[i].set_ylabel('dB')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Fakeprint per Algorithm (1-8 kHz)', fontsize=13, y=1.01)
plt.tight_layout()
save_plot('03a_fakeprint_per_algorithm')

# Overlay top algorithms on same plot
fig, ax = plt.subplots(figsize=(16, 6))
top_algos = df.loc[ai_mask, 'algorithm'].value_counts().head(8).index
for algo in top_algos:
    mask = (df['algorithm'] == algo).values & ai_mask
    if mask.sum() == 0: continue
    ax.plot(freq_axis, np.mean(X_fakeprint[mask], axis=0), label=f'{algo} (n={mask.sum()})', alpha=0.8)
ax.plot(freq_axis, fp_real_mean, 'k--', linewidth=2, label='Real baseline')
ax.set_title('Fakeprint Comparison: Top Algorithms')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Peak Height (dB)')
ax.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
save_plot('03a_fakeprint_overlay_top_algos')

Algorithms found: ['chirp-v3.5', 'chirp-v3', 'chirp-v2-xxl-alpha', 'udio-120s', 'udio-30s', 'MusicGen_medium', 'audioldm2', 'musicldm', 'mustango', 'stable_audio_open', 'acestep', 'audioldm', 'brev', 'diffrhythm', 'mubert', 'producer', 'songgen', 'suno', 'udio', 'stableaudio']


In [98]:
# === 3b. AI detection features per algorithm ===
df_ai_algo = df_ai[ai_mask].copy()
df_ai_algo['algorithm'] = df.loc[ai_mask, 'algorithm'].values

key_feats = ['PCI', 'HNR', 'Spectral Periodicity', 'Artifact Energy', 'Phase Dev Mean', 'Max PTB']
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.ravel()
for i, feat in enumerate(key_feats):
    top_a = df_ai_algo['algorithm'].value_counts().head(8).index
    sub = df_ai_algo[df_ai_algo['algorithm'].isin(top_a)]
    sns.boxplot(data=sub, x='algorithm', y=feat, hue='algorithm', legend=False, ax=axes[i])
    axes[i].set_title(feat)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')
plt.suptitle('AI Detection Features by Algorithm', fontsize=13)
plt.tight_layout()
save_plot('03b_ai_features_per_algorithm')

# Kruskal-Wallis test: do algorithms differ?
print('Kruskal-Wallis test (across algorithms):')
for feat in ai_feat_names:
    groups = [g[feat].dropna().values for _, g in df_ai_algo.groupby('algorithm') if len(g) > 5]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {feat:25s}: H={stat:.1f}, p={p:.2e} {sig}')

Kruskal-Wallis test (across algorithms):
  PCI                      : H=2245.6, p=0.00e+00 ***
  Phase Dev Mean           : H=6010.9, p=0.00e+00 ***
  Phase Dev Std            : H=7666.3, p=0.00e+00 ***
  IF Stability Mean        : H=3876.3, p=0.00e+00 ***
  IF Stability Std         : H=2266.6, p=0.00e+00 ***
  GD Dev Mean              : H=1311.5, p=1.19e-266 ***
  GD Dev Std               : H=2098.8, p=0.00e+00 ***
  HNR                      : H=1665.2, p=0.00e+00 ***
  Flatness Mean            : H=2465.7, p=0.00e+00 ***
  Flatness Std             : H=3866.3, p=0.00e+00 ***
  Rolloff Ratio            : H=1460.4, p=1.43e-298 ***
  SSM Novelty              : H=1967.5, p=0.00e+00 ***
  PTB 2x                   : H=6787.4, p=0.00e+00 ***
  PTB 4x                   : H=5289.6, p=0.00e+00 ***
  PTB 8x                   : H=5291.1, p=0.00e+00 ***
  PeakCnt 2x               : H=5210.2, p=0.00e+00 ***
  PeakCnt 4x               : H=4453.1, p=0.00e+00 ***
  PeakCnt 8x               : H=4581.6, 

In [99]:
# === 3c. Classical features per algorithm ===
df_cls_algo = df_cls[ai_mask].copy()
df_cls_algo['algorithm'] = df.loc[ai_mask, 'algorithm'].values

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.ravel()
for i, col in enumerate(mean_cols):
    top_a = df_cls_algo['algorithm'].value_counts().head(8).index
    sub = df_cls_algo[df_cls_algo['algorithm'].isin(top_a)]
    sns.boxplot(data=sub, x='algorithm', y=col, hue='algorithm', legend=False, ax=axes[i])
    axes[i].set_title(col.replace('_mean', ''))
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')
plt.suptitle('Classical Features by Algorithm (AI songs)', fontsize=13)
plt.tight_layout()
save_plot('03c_classical_per_algorithm')

print('Per-algorithm analysis complete.')

Per-algorithm analysis complete.


## 4. SMP Dataset: Real Plagiarism Pairs

In [100]:
# === 4. SMP pairs: feature similarity between originals and comparisons ===
smp_mask = df['dataset'] == 'smp'
df_smp = df[smp_mask].copy()

if len(df_smp) > 0 and 'pair_number' in df_smp.columns:
    # Compute cosine similarity within pairs
    pair_sims_classical = []
    pair_sims_ai = []
    pair_labels = []
    
    smp_idx = np.where(smp_mask)[0]
    
    for pn in df_smp['pair_number'].dropna().unique():
        pair_rows = df_smp[df_smp['pair_number'] == pn]
        if len(pair_rows) < 2:
            continue
        
        idx_a = pair_rows.index[0]
        idx_b = pair_rows.index[1]
        
        cs = 1 - cosine(X_classical[idx_a], X_classical[idx_b])
        ai_s = 1 - cosine(X_ai_det[idx_a], X_ai_det[idx_b])
        
        pair_sims_classical.append(cs)
        pair_sims_ai.append(ai_s)
        pair_labels.append(pair_rows.iloc[0].get('relation', 'unknown'))
    
    if pair_sims_classical:
        df_smp_sim = pd.DataFrame({
            'classical_sim': pair_sims_classical,
            'ai_det_sim': pair_sims_ai,
            'relation': pair_labels
        })
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(data=df_smp_sim, x='classical_sim', hue='relation', kde=True, ax=axes[0])
        axes[0].set_title('SMP Pairs: Classical Feature Cosine Similarity')
        
        sns.histplot(data=df_smp_sim, x='ai_det_sim', hue='relation', kde=True, ax=axes[1])
        axes[1].set_title('SMP Pairs: AI Detection Feature Cosine Similarity')
        plt.tight_layout()
        save_plot('04_smp_pair_similarity')
        
        print('SMP pair similarity stats:')
        print(df_smp_sim.groupby('relation').agg(['mean', 'std']).round(3))
    
    # SMP classical feature distribution
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()
    df_smp_cls = df_cls[smp_mask]
    for i, col in enumerate(mean_cols):
        if i >= len(axes): break
        sns.histplot(df_smp_cls[col], kde=True, ax=axes[i], color='green')
        axes[i].set_title(f'SMP: {col.replace("_mean", "")}')
    plt.suptitle('SMP Dataset: Classical Feature Distributions', fontsize=13)
    plt.tight_layout()
    save_plot('04_smp_classical_distributions')
else:
    print('SMP dataset not found in features or missing pair_number column')

SMP pair similarity stats:
           classical_sim        ai_det_sim       
                    mean    std       mean    std
relation                                         
plag               0.996  0.002      0.997  0.003
plag_doubt         0.994  0.003      0.997  0.003
remake             0.997  0.002      0.997  0.003


## 5. FakeMusicCaps: Fake Song Patterns

In [101]:
# === 5. FakeMusicCaps: common patterns in AI-generated music ===
fmc_mask = df['dataset'] == 'fakemusiccaps'
df_fmc = df[fmc_mask].copy()

if len(df_fmc) > 0:
    # Fakeprint by generator folder
    generators = df_fmc['subset'].unique()
    print(f'FakeMusicCaps generators: {sorted(generators)}')
    
    fig, ax = plt.subplots(figsize=(16, 6))
    for gen in sorted(generators):
        mask = (df['subset'] == gen).values & fmc_mask.values
        if mask.sum() == 0: continue
        ax.plot(freq_axis, np.mean(X_fakeprint[mask], axis=0),
                label=f'{gen} (n={mask.sum()})', alpha=0.8)
    ax.plot(freq_axis, fp_real_mean, 'k--', linewidth=2, label='Real baseline')
    ax.set_title('FakeMusicCaps: Fakeprint by Generator')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('dB')
    ax.legend(bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    save_plot('05_fmc_fakeprint_by_generator')
    
    # AI detection features by generator
    df_fmc_ai = df_ai[fmc_mask.values].copy()
    df_fmc_ai['generator'] = df_fmc['subset'].values
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes = axes.ravel()
    for i, feat in enumerate(key_feats):
        if i >= len(axes): break
        sns.boxplot(data=df_fmc_ai, x='generator', y=feat, hue='generator', legend=False, ax=axes[i])
        axes[i].set_title(feat)
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].set_xlabel('')
    plt.suptitle('FakeMusicCaps: AI Detection Features by Generator', fontsize=13)
    plt.tight_layout()
    save_plot('05_fmc_ai_features_by_generator')
    
    # Classical features by generator
    df_fmc_cls = df_cls[fmc_mask.values].copy()
    df_fmc_cls['generator'] = df_fmc['subset'].values
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes = axes.ravel()
    for i, col in enumerate(mean_cols):
        if i >= len(axes): break
        sns.boxplot(data=df_fmc_cls, x='generator', y=col, hue='generator', legend=False, ax=axes[i])
        axes[i].set_title(col.replace('_mean', ''))
        axes[i].tick_params(axis='x', rotation=45)
    plt.suptitle('FakeMusicCaps: Classical Features by Generator', fontsize=13)
    plt.tight_layout()
    save_plot('05_fmc_classical_by_generator')
    
    print('FakeMusicCaps feature analysis complete.')
else:
    print('FakeMusicCaps not found in features.')

FakeMusicCaps generators: ['MusicGen_medium', 'audioldm2', 'musicldm', 'mustango', 'stable_audio_open']
FakeMusicCaps feature analysis complete.


In [102]:
# === 5b. FakeMusicCaps: Same-Prompt Pairs (same YouTube ID, different generators) ===
# FMC filenames follow {generator}_{youtube_id}.wav — tracks sharing a YouTube ID
# were generated from the same MusicCaps caption (same prompt, different model).

fmc_mask = (df['dataset'] == 'fakemusiccaps').values
df_fmc = df[fmc_mask].copy().reset_index(drop=True)
X_cls_fmc = X_classical[fmc_mask]
X_fp_fmc = X_fakeprint[fmc_mask]

if len(df_fmc) == 0:
    print('FakeMusicCaps not found in feature cache.')
else:
    def extract_yt_id(row):
        stem = Path(row['filename']).stem  # strip .wav
        prefix = str(row['subset']) + '_'
        return stem[len(prefix):] if stem.startswith(prefix) else stem

    df_fmc['youtube_id'] = df_fmc.apply(extract_yt_id, axis=1)
    yt_groups = df_fmc.groupby('youtube_id')
    multi_gen = {yt: grp for yt, grp in yt_groups if len(grp) > 1}
    n_same_pairs = sum(len(g) * (len(g) - 1) // 2 for g in multi_gen.values())

    print(f'FMC tracks loaded:              {len(df_fmc)}')
    print(f'Unique YouTube IDs:             {df_fmc["youtube_id"].nunique()}')
    print(f'YouTube IDs with >1 generator:  {len(multi_gen)}')
    print(f'Total same-prompt pairs:        {n_same_pairs}')

    # --- Coverage: how many generators cover each YouTube ID ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    gen_per_yt = df_fmc.groupby('youtube_id')['subset'].nunique()
    vc = gen_per_yt.value_counts().sort_index()
    axes[0].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='white')
    for i, (x, y) in enumerate(zip(vc.index, vc.values)):
        axes[0].text(i, y + 0.5, str(y), ha='center', va='bottom', fontsize=9)
    axes[0].set_title('Number of Generators per YouTube ID (same-prompt coverage)')
    axes[0].set_xlabel('Generators covering the same prompt')
    axes[0].set_ylabel('Count of YouTube IDs')

    gen_in_pairs = {}
    for _, grp in multi_gen.items():
        for gen in grp['subset']:
            gen_in_pairs[gen] = gen_in_pairs.get(gen, 0) + 1
    if gen_in_pairs:
        s = pd.Series(gen_in_pairs).sort_values(ascending=True)
        axes[1].barh(s.index, s.values, color='coral', edgecolor='white')
        axes[1].set_title('Tracks in Same-Prompt Pairs by Generator')
        axes[1].set_xlabel('Number of tracks')

    plt.suptitle('FakeMusicCaps: Same-Prompt Pair Coverage', fontsize=13)
    plt.tight_layout()
    save_plot('05b_fmc_same_prompt_coverage')

    # --- Feature similarity: same-prompt vs different-prompt ---
    from itertools import combinations
    from scipy.spatial.distance import cosine as cos_dist

    same_cls, same_fp = [], []
    diff_cls, diff_fp = [], []
    yt_ids = df_fmc['youtube_id'].values

    for yt, grp in df_fmc.groupby('youtube_id'):
        idxs = grp.index.tolist()
        for i, j in combinations(idxs, 2):
            same_cls.append(1 - cos_dist(X_cls_fmc[i], X_cls_fmc[j]))
            same_fp.append(1 - cos_dist(X_fp_fmc[i], X_fp_fmc[j]))

    rng = np.random.default_rng(42)
    target = min(max(len(same_cls) * 5, 500), 3000)
    attempts = 0
    while len(diff_cls) < target and attempts < target * 20:
        i, j = rng.integers(0, len(df_fmc), size=2)
        if yt_ids[i] != yt_ids[j]:
            diff_cls.append(1 - cos_dist(X_cls_fmc[i], X_cls_fmc[j]))
            diff_fp.append(1 - cos_dist(X_fp_fmc[i], X_fp_fmc[j]))
        attempts += 1

    if same_cls and diff_cls:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        for ax, same, diff, title in [
            (axes[0], same_cls, diff_cls, 'Classical Features'),
            (axes[1], same_fp, diff_fp, 'Fakeprint'),
        ]:
            ax.hist(diff, bins=40, alpha=0.55, label=f'Different prompt (n={len(diff)})',
                    color='gray', density=True)
            ax.hist(same, bins=40, alpha=0.70, label=f'Same prompt / same YT ID (n={len(same)})',
                    color='steelblue', density=True)
            ax.axvline(np.median(same), color='steelblue', linestyle='--', lw=1.5,
                       label=f'Median same = {np.median(same):.3f}')
            ax.axvline(np.median(diff), color='gray', linestyle='--', lw=1.5,
                       label=f'Median diff = {np.median(diff):.3f}')
            ax.set_xlabel('Cosine Similarity')
            ax.set_ylabel('Density')
            ax.set_title(f'FMC {title}: Same vs Different Prompt')
            ax.legend(fontsize=8)
        plt.suptitle('FakeMusicCaps: Same-Prompt vs Different-Prompt Feature Similarity', fontsize=13)
        plt.tight_layout()
        save_plot('05b_fmc_same_prompt_similarity')

        print(f'\nClassical:  same-prompt median={np.median(same_cls):.3f}  |  diff-prompt median={np.median(diff_cls):.3f}')
        print(f'Fakeprint:  same-prompt median={np.median(same_fp):.3f}  |  diff-prompt median={np.median(diff_fp):.3f}')
    else:
        print(f'Found {len(same_cls)} same-prompt pairs — run after feature cache is populated for similarity analysis.')


FMC tracks loaded:              2000
Unique YouTube IDs:             400
YouTube IDs with >1 generator:  400
Total same-prompt pairs:        4000

Classical:  same-prompt median=0.983  |  diff-prompt median=0.979
Fakeprint:  same-prompt median=0.702  |  diff-prompt median=0.700


## 6. SONICS Lyric Pairs: What Paired Tracks Share

In [103]:
# === 6. SONICS lyric pairs: real vs fake matched by lyrics ===
lyric_map_path = Path('../data/sonics/lyric_pairs_mapping.csv')
if lyric_map_path.exists():
    df_lmap = pd.read_csv(lyric_map_path)
    
    # Build filename -> feature index mapping for sonics pair subsets
    fake_mask = (df['dataset'] == 'sonics') & (df['subset'] == 'lyric_pair_fake')
    real_mask = (df['dataset'] == 'sonics') & (df['subset'] == 'lyric_pair_real')
    
    # Map both with and without .wav extension
    fake_idx_map = {}
    for idx, row in df.loc[fake_mask].iterrows():
        fn = row['filename']
        fake_idx_map[fn] = idx
        if fn.endswith('.wav'):
            fake_idx_map[fn[:-4]] = idx
        else:
            fake_idx_map[fn + '.wav'] = idx
    real_idx_map = {}
    for idx, row in df.loc[real_mask].iterrows():
        # Map by youtube_id or filename
        real_idx_map[row['filename']] = idx
    
    # Compute pairwise similarities
    pair_results = []
    for _, pm in df_lmap.iterrows():
        fake_fn = pm['fake_filename']
        real_yt = pm['real_youtube_id']
        
        fi = fake_idx_map.get(fake_fn)
        # Try matching real by youtube_id prefix in filename
        ri = None
        for fn, idx in real_idx_map.items():
            if real_yt in str(fn):
                ri = idx
                break
        
        if fi is None or ri is None:
            continue
        
        cs_cls = 1 - cosine(X_classical[fi], X_classical[ri]) if np.any(X_classical[fi]) else np.nan
        cs_fp = 1 - cosine(X_fakeprint[fi], X_fakeprint[ri]) if np.any(X_fakeprint[fi]) else np.nan
        
        pair_results.append({
            'lyric_sim': pm.get('similarity', np.nan),
            'classical_sim': cs_cls,
            'fakeprint_sim': cs_fp,
            'algorithm': df.loc[fi, 'algorithm'] if fi in df.index else 'unknown',
        })
    
    if pair_results:
        df_pairs = pd.DataFrame(pair_results)
        print(f'Matched lyric pairs: {len(df_pairs)}')
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        sns.histplot(df_pairs['classical_sim'].dropna(), kde=True, ax=axes[0], color='teal')
        axes[0].set_title('Lyric Pairs: Classical Cosine Similarity')
        axes[0].axvline(df_pairs['classical_sim'].median(), color='red', linestyle='--')
        
        sns.histplot(df_pairs['fakeprint_sim'].dropna(), kde=True, ax=axes[1], color='orange')
        axes[1].set_title('Lyric Pairs: Fakeprint Cosine Similarity')
        axes[1].axvline(df_pairs['fakeprint_sim'].median(), color='red', linestyle='--')
        
        sns.scatterplot(data=df_pairs, x='classical_sim', y='fakeprint_sim',
                       hue='algorithm', alpha=0.6, ax=axes[2])
        axes[2].set_title('Classical vs Fakeprint Similarity')
        axes[2].legend(bbox_to_anchor=(1.05, 1), fontsize=8)
        plt.tight_layout()
        save_plot('06_sonics_lyric_pair_similarity')
        
        print(f'Classical sim: mean={df_pairs["classical_sim"].mean():.3f}, median={df_pairs["classical_sim"].median():.3f}')
        print(f'Fakeprint sim: mean={df_pairs["fakeprint_sim"].mean():.3f}, median={df_pairs["fakeprint_sim"].median():.3f}')
        print('(High classical + low fakeprint = similar song content, different generation artifacts)')
    else:
        print('Could not match lyric pairs to feature cache.')
else:
    print('Lyric pairs mapping not found.')

Matched lyric pairs: 462
Classical sim: mean=0.990, median=0.992
Fakeprint sim: mean=0.356, median=0.354
(High classical + low fakeprint = similar song content, different generation artifacts)


## 7. Genre-Based Analysis

In [104]:
# === 7. Common patterns per genre (AI and real) ===
has_genre = df['genre'].notna() & (df['genre'] != '')
df_genre = df[has_genre].copy()

if len(df_genre) > 50:
    top_genres = df_genre['genre'].value_counts().head(10).index
    genre_mask = df_genre['genre'].isin(top_genres)
    df_genre_top = df_genre[genre_mask]
    
    # Classical features per genre
    df_genre_cls = df_cls[has_genre.values][genre_mask.values].copy()
    df_genre_cls['genre'] = df_genre_top['genre'].values
    df_genre_cls['is_ai'] = df_genre_top['is_ai'].values
    
    for col in mean_cols:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.boxplot(data=df_genre_cls, x='genre', y=col, hue='is_ai', ax=ax)
        ax.set_title(f'{col.replace("_mean", "")}: Genre x Real/AI')
        ax.tick_params(axis='x', rotation=45)
        ax.legend(title='Is AI', labels=['Real', 'AI'])
        plt.tight_layout()
        save_plot(f'07_genre_{col.replace("_mean", "").lower().replace(" ", "_")}')
    
    # AI detection features per genre
    df_genre_ai = df_ai[has_genre.values][genre_mask.values].copy()
    df_genre_ai['genre'] = df_genre_top['genre'].values
    df_genre_ai['is_ai'] = df_genre_top['is_ai'].values
    
    for feat in ['PCI', 'HNR', 'Artifact Energy']:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.boxplot(data=df_genre_ai, x='genre', y=feat, hue='is_ai', ax=ax)
        ax.set_title(f'{feat}: Genre x Real/AI')
        ax.tick_params(axis='x', rotation=45)
        ax.legend(title='Is AI', labels=['Real', 'AI'])
        plt.tight_layout()
        save_plot(f'07_genre_ai_{feat.lower().replace(" ", "_")}')
    
    # Genre centroid similarity heatmap (classical features)
    genre_centroids = {}
    genre_idx = has_genre.values
    for g in top_genres:
        mask = (df['genre'] == g).values & genre_idx
        if mask.sum() > 0:
            genre_centroids[g] = np.mean(X_classical[mask], axis=0)
    
    if len(genre_centroids) > 1:
        genres_list = list(genre_centroids.keys())
        sim_matrix = np.zeros((len(genres_list), len(genres_list)))
        for i, g1 in enumerate(genres_list):
            for j, g2 in enumerate(genres_list):
                sim_matrix[i, j] = 1 - cosine(genre_centroids[g1], genre_centroids[g2])
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(sim_matrix, xticklabels=genres_list, yticklabels=genres_list,
                   annot=True, fmt='.2f', cmap='RdYlBu_r', ax=ax)
        ax.set_title('Genre Centroid Similarity (Classical Features)')
        plt.tight_layout()
        save_plot('07_genre_similarity_heatmap')
    
    print('Genre analysis complete.')
else:
    print('Not enough genre-labeled data.')

Genre analysis complete.


## 8. Echoes Dataset: Generated vs Original Pairs

In [105]:
# === 8a. Echoes: What generated songs have in common ===
echoes_gen_mask = (df['dataset'] == 'echoes') & (df['subset'] == 'generated')
echoes_ori_mask = (df['dataset'] == 'echoes') & (df['subset'] == 'original')

if echoes_gen_mask.sum() > 0:
    # Fakeprint by Echoes generator
    echoes_algos = df.loc[echoes_gen_mask, 'algorithm'].unique()
    print(f'Echoes generators: {sorted(echoes_algos)}')
    
    fig, ax = plt.subplots(figsize=(16, 6))
    for gen in sorted(echoes_algos):
        mask = (df['algorithm'] == gen).values & echoes_gen_mask.values
        if mask.sum() == 0: continue
        ax.plot(freq_axis, np.mean(X_fakeprint[mask], axis=0),
                label=f'{gen} (n={mask.sum()})', alpha=0.8)
    if echoes_ori_mask.sum() > 0:
        ax.plot(freq_axis, np.mean(X_fakeprint[echoes_ori_mask], axis=0),
                'k--', linewidth=2, label=f'Originals (n={echoes_ori_mask.sum()})')
    ax.set_title('Echoes: Fakeprint by Generator')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('dB')
    ax.legend(bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    save_plot('08a_echoes_fakeprint_by_generator')
    
    # AI detection features: generated vs original
    df_echoes_ai = df_ai[echoes_gen_mask.values | echoes_ori_mask.values].copy()
    df_echoes_ai['type'] = np.where(
        df.loc[echoes_gen_mask.values | echoes_ori_mask.values, 'subset'].values == 'generated',
        'Generated', 'Original'
    )
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes = axes.ravel()
    for i, feat in enumerate(key_feats):
        if i >= len(axes): break
        sns.boxplot(data=df_echoes_ai, x='type', y=feat, hue='type', legend=False, ax=axes[i])
        axes[i].set_title(feat)
    plt.suptitle('Echoes: Generated vs Original (AI Detection Features)', fontsize=13)
    plt.tight_layout()
    save_plot('08a_echoes_ai_features_gen_vs_ori')
    
    # TTA vs ATA comparison
    if 'echoes_type' in df.columns:
        echoes_type_mask = echoes_gen_mask & df['echoes_type'].notna()
        if echoes_type_mask.sum() > 0:
            df_etype = df_ai[echoes_type_mask.values].copy()
            df_etype['echoes_type'] = df.loc[echoes_type_mask, 'echoes_type'].values
            
            fig, axes = plt.subplots(2, 3, figsize=(18, 10))
            axes = axes.ravel()
            for i, feat in enumerate(key_feats):
                if i >= len(axes): break
                sns.boxplot(data=df_etype, x='echoes_type', y=feat,
                           hue='echoes_type', legend=False, ax=axes[i])
                axes[i].set_title(feat)
            plt.suptitle('Echoes: TTA vs ATA (AI Detection Features)', fontsize=13)
            plt.tight_layout()
            save_plot('08a_echoes_tta_vs_ata')

    print('Echoes generated analysis complete.')
else:
    print('No Echoes generated data found.')

Echoes generators: ['acestep', 'audioldm', 'brev', 'diffrhythm', 'mubert', 'producer', 'songgen', 'stableaudio', 'suno', 'udio']
Echoes generated analysis complete.


In [106]:
# === 8b. Echoes pairs: generated vs original similarity ===
if echoes_gen_mask.sum() > 0 and echoes_ori_mask.sum() > 0 and 'original_audio' in df.columns:
    # Build original_audio -> index mapping
    ori_map = {}
    for idx, row in df[echoes_ori_mask].iterrows():
        oa = row.get('original_audio', '')
        if pd.notna(oa) and oa:
            ori_map[oa] = idx
    
    # Match generated to originals
    echo_pair_results = []
    for idx, row in df[echoes_gen_mask].iterrows():
        oa = row.get('original_audio', '')
        if pd.isna(oa) or oa not in ori_map:
            continue
        oi = ori_map[oa]
        
        cs = 1 - cosine(X_classical[idx], X_classical[oi]) if np.any(X_classical[idx]) else np.nan
        fps = 1 - cosine(X_fakeprint[idx], X_fakeprint[oi]) if np.any(X_fakeprint[idx]) else np.nan
        
        echo_pair_results.append({
            'classical_sim': cs,
            'fakeprint_sim': fps,
            'generator': row['algorithm'],
            'echoes_type': row.get('echoes_type', ''),
        })
    
    if echo_pair_results:
        df_echo_pairs = pd.DataFrame(echo_pair_results)
        print(f'Matched Echoes pairs: {len(df_echo_pairs)}')
        
        fig, axes = plt.subplots(1, 3, figsize=(20, 5))
        
        sns.histplot(df_echo_pairs['classical_sim'].dropna(), kde=True, ax=axes[0], color='teal')
        axes[0].set_title('Echoes Pairs: Classical Similarity')
        
        sns.boxplot(data=df_echo_pairs, x='generator', y='classical_sim',
                   hue='generator', legend=False, ax=axes[1])
        axes[1].set_title('Classical Sim by Generator')
        axes[1].tick_params(axis='x', rotation=45)
        
        sns.scatterplot(data=df_echo_pairs, x='classical_sim', y='fakeprint_sim',
                       hue='generator', alpha=0.5, ax=axes[2])
        axes[2].set_title('Classical vs Fakeprint Sim')
        axes[2].legend(bbox_to_anchor=(1.05, 1), fontsize=8)
        plt.tight_layout()
        save_plot('08b_echoes_pair_similarity')
        
        # TTA vs ATA similarity
        if 'echoes_type' in df_echo_pairs.columns:
            print('\nSimilarity by generation type:')
            print(df_echo_pairs.groupby('echoes_type')[['classical_sim', 'fakeprint_sim']].agg(['mean', 'std']).round(3))
        
        print('\nSimilarity by generator:')
        print(df_echo_pairs.groupby('generator')[['classical_sim', 'fakeprint_sim']].agg(['mean', 'std']).round(3))
else:
    print('Cannot compute Echoes pairs (missing data or original_audio column).')

Matched Echoes pairs: 3577

Similarity by generation type:
            classical_sim        fakeprint_sim       
                     mean    std          mean    std
echoes_type                                          
ATA                 0.980  0.018         0.529  0.330
TTA                 0.981  0.014         0.714  0.281

Similarity by generator:
            classical_sim        fakeprint_sim       
                     mean    std          mean    std
generator                                            
acestep             0.983  0.010         0.824  0.100
audioldm            0.984  0.012         0.257  0.097
brev                0.986  0.008         0.872  0.111
diffrhythm          0.967  0.023         0.877  0.114
mubert              0.982  0.010         0.829  0.137
producer            0.986  0.011         0.876  0.095
songgen             0.982  0.010         0.227  0.076
stableaudio         0.979  0.013         0.838  0.123
suno                0.985  0.010         0.877  0.1

## 9. Beyond AI Artifacts: Musical Distinguishing Features

In [107]:
# === 9. Feature importance: what distinguishes beyond artifacts? ===
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 9a. Classical features only — can we tell real from AI without artifact features?
valid = ~np.isnan(X_classical).any(axis=1) & ~np.isinf(X_classical).any(axis=1)
X_cls_valid = np.nan_to_num(X_classical[valid])
y_valid = df['is_ai'].values[valid].astype(int)

if len(np.unique(y_valid)) == 2 and len(y_valid) > 100:
    # Subsample for speed
    n_sub = min(5000, len(y_valid))
    idx_sub = np.random.choice(len(y_valid), n_sub, replace=False)
    
    rf_cls = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    scores_cls = cross_val_score(rf_cls, X_cls_valid[idx_sub], y_valid[idx_sub], cv=5, scoring='roc_auc')
    print(f'Classical features only — AUC: {scores_cls.mean():.3f} +/- {scores_cls.std():.3f}')
    
    # 9b. AI detection features
    X_ai_valid = np.nan_to_num(X_ai_det[valid])
    rf_ai = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    scores_ai = cross_val_score(rf_ai, X_ai_valid[idx_sub], y_valid[idx_sub], cv=5, scoring='roc_auc')
    print(f'AI detection features — AUC: {scores_ai.mean():.3f} +/- {scores_ai.std():.3f}')
    
    # 9c. Fakeprint
    X_fp_valid = np.nan_to_num(X_fakeprint[valid])
    rf_fp = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    scores_fp = cross_val_score(rf_fp, X_fp_valid[idx_sub], y_valid[idx_sub], cv=5, scoring='roc_auc')
    print(f'Fakeprint features — AUC: {scores_fp.mean():.3f} +/- {scores_fp.std():.3f}')
    
    # 9d. All combined
    X_all = np.hstack([X_cls_valid, X_ai_valid, X_fp_valid])
    rf_all = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    scores_all = cross_val_score(rf_all, X_all[idx_sub], y_valid[idx_sub], cv=5, scoring='roc_auc')
    print(f'All features combined — AUC: {scores_all.mean():.3f} +/- {scores_all.std():.3f}')
    
    # Feature importance from classical RF
    rf_cls.fit(X_cls_valid[idx_sub], y_valid[idx_sub])
    importances = rf_cls.feature_importances_
    
    # Group importances
    group_imp = {}
    for gname, (start, end) in CLASSICAL_GROUPS.items():
        group_imp[gname] = importances[start:end].sum()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # AUC comparison bar chart
    auc_data = pd.DataFrame({
        'Feature Set': ['Classical', 'AI Detection', 'Fakeprint', 'All Combined'],
        'AUC': [scores_cls.mean(), scores_ai.mean(), scores_fp.mean(), scores_all.mean()],
        'Std': [scores_cls.std(), scores_ai.std(), scores_fp.std(), scores_all.std()],
    })
    axes[0].bar(auc_data['Feature Set'], auc_data['AUC'], yerr=auc_data['Std'],
               capsize=5, color=['teal', 'orange', 'red', 'purple'])
    axes[0].set_title('Real vs AI Classification AUC by Feature Set')
    axes[0].set_ylabel('ROC AUC')
    axes[0].set_ylim(0.5, 1.05)
    axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    
    # Group importance
    gi_df = pd.DataFrame(list(group_imp.items()), columns=['Group', 'Importance'])
    gi_df = gi_df.sort_values('Importance', ascending=True)
    axes[1].barh(gi_df['Group'], gi_df['Importance'], color='teal')
    axes[1].set_title('Classical Feature Group Importance (for AI detection)')
    axes[1].set_xlabel('Sum of Feature Importances')
    plt.tight_layout()
    save_plot('09_feature_set_comparison')
    
    print('\nClassical feature groups ranked by importance for AI detection:')
    for _, row in gi_df.sort_values('Importance', ascending=False).iterrows():
        print(f'  {row["Group"]:25s}: {row["Importance"]:.4f}')
    print('\n(Higher classical importance = musical patterns that differ between real and AI,')
    print(' even without explicit artifact features. These are "beyond artifact" signals.)')
else:
    print('Not enough data for classification analysis.')

Classical features only — AUC: 0.986 +/- 0.005
AI detection features — AUC: 0.992 +/- 0.002
Fakeprint features — AUC: 0.992 +/- 0.001
All features combined — AUC: 0.998 +/- 0.000

Classical feature groups ranked by importance for AI detection:
  Mel-spectrogram          : 0.5921
  MFCC                     : 0.1652
  Temporal                 : 0.1169
  Spectral Contrast        : 0.0992
  Chroma                   : 0.0180
  Tonnetz                  : 0.0086

(Higher classical importance = musical patterns that differ between real and AI,
 even without explicit artifact features. These are "beyond artifact" signals.)


## 10. Cross-Dataset Analysis

In [108]:
# === 10a. Cross-dataset embedding space (PCA) ===
# Use classical features for cross-dataset comparison
X_cls_scaled = StandardScaler().fit_transform(np.nan_to_num(X_classical))
pca_cls = PCA(n_components=2)
X_pca_cls = pca_cls.fit_transform(X_cls_scaled)

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# By dataset
for ds in df['dataset'].unique():
    mask = df['dataset'] == ds
    axes[0].scatter(X_pca_cls[mask, 0], X_pca_cls[mask, 1], alpha=0.3, s=8, label=ds)
axes[0].set_title(f'Classical Features PCA (var: {pca_cls.explained_variance_ratio_.sum():.1%})')
axes[0].legend()
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# By real/AI
for label, color, name in [(True, 'red', 'AI'), (False, 'blue', 'Real')]:
    mask = df['is_ai'] == label
    axes[1].scatter(X_pca_cls[mask, 0], X_pca_cls[mask, 1], c=color, alpha=0.3, s=8, label=name)
axes[1].set_title('Classical Features PCA (Real vs AI)')
axes[1].legend()

# By dataset + AI/real combined
combos = []
for ds in df['dataset'].unique():
    for is_ai in [True, False]:
        mask = (df['dataset'] == ds) & (df['is_ai'] == is_ai)
        if mask.sum() > 0:
            label = f'{ds} ({"AI" if is_ai else "Real"})'
            axes[2].scatter(X_pca_cls[mask, 0], X_pca_cls[mask, 1], alpha=0.3, s=8, label=label)
axes[2].set_title('Classical Features: Dataset x AI/Real')
axes[2].legend(bbox_to_anchor=(1.05, 1), fontsize=8)
plt.tight_layout()
save_plot('10a_cross_dataset_pca_classical')

print('Cross-dataset PCA complete.')

Cross-dataset PCA complete.


In [109]:
# === 10b. Cross-dataset feature statistics comparison ===
# Compare key features across datasets
summary_rows = []
for ds in df['dataset'].unique():
    for is_ai in [True, False]:
        mask = (df['dataset'] == ds) & (df['is_ai'] == is_ai)
        if mask.sum() < 3:
            continue
        idx = mask.values
        row = {
            'dataset': ds,
            'type': 'AI' if is_ai else 'Real',
            'n': mask.sum(),
        }
        # Classical group means
        for gname, (start, end) in CLASSICAL_GROUPS.items():
            row[f'{gname}_mean'] = np.mean(X_classical[idx, start:end])
        # AI detection means
        for gname, (start, end) in AI_DETECTION_GROUPS.items():
            row[f'{gname}_mean'] = np.mean(X_ai_det[idx, start:end])
        # Fakeprint summary
        row['fakeprint_energy'] = np.mean(fp_energy[idx])
        row['fakeprint_max'] = np.mean(fp_max[idx])
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print('Cross-Dataset Feature Summary:')
print(df_summary.to_string(index=False))

# Heatmap of key stats
feature_cols = [c for c in df_summary.columns if c.endswith('_mean') or 'fakeprint' in c]
df_summary['label'] = df_summary['dataset'] + ' (' + df_summary['type'] + ')'
pivot = df_summary.set_index('label')[feature_cols]

# Normalize for heatmap
pivot_norm = (pivot - pivot.mean()) / (pivot.std() + 1e-10)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot_norm, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Cross-Dataset Feature Comparison (z-scored)')
plt.tight_layout()
save_plot('10b_cross_dataset_heatmap')

print('\nCross-dataset analysis complete.')

Cross-Dataset Feature Summary:
      dataset type    n  MFCC_mean  Mel-spectrogram_mean  Chroma_mean  Tonnetz_mean  Spectral Contrast_mean  Temporal_mean  Phase Continuity_mean  Spectral AI Indicators_mean  Fourier Artifact Summary_mean  fakeprint_energy  fakeprint_max
       sonics   AI 2962 -42.941727              2.817713     0.307566      0.067926               13.830605      20.739174               1.469980                     0.767963                       6.691352       3260.669189      55.952892
       sonics Real 1053 -40.433010              3.751621     0.326062      0.060850               11.356504      20.776125               1.474362                     0.239327                       3.552395       2059.901123      11.985811
fakemusiccaps   AI 2000 -46.063671              1.988004     0.286641      0.067889               11.871716      21.605471               1.466621                     0.690520                       6.237629       4145.759766      22.080173
          smp

In [110]:
# === 10c. Embedding space analysis (if available) ===
if has_mert and has_clap:
    # MERT PCA
    X_mert_scaled = StandardScaler().fit_transform(np.nan_to_num(X_mert))
    X_clap_scaled = StandardScaler().fit_transform(np.nan_to_num(X_clap))
    
    pca_mert = PCA(n_components=2).fit_transform(X_mert_scaled)
    pca_clap = PCA(n_components=2).fit_transform(X_clap_scaled)
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    
    for label, color, name in [(True, 'red', 'AI'), (False, 'blue', 'Real')]:
        mask = df['is_ai'] == label
        axes[0, 0].scatter(pca_mert[mask, 0], pca_mert[mask, 1], c=color, alpha=0.3, s=8, label=name)
        axes[0, 1].scatter(pca_clap[mask, 0], pca_clap[mask, 1], c=color, alpha=0.3, s=8, label=name)
    axes[0, 0].set_title('MERT Embeddings PCA (Real vs AI)')
    axes[0, 0].legend()
    axes[0, 1].set_title('CLAP Embeddings PCA (Real vs AI)')
    axes[0, 1].legend()
    
    for ds in df['dataset'].unique():
        mask = df['dataset'] == ds
        axes[1, 0].scatter(pca_mert[mask, 0], pca_mert[mask, 1], alpha=0.3, s=8, label=ds)
        axes[1, 1].scatter(pca_clap[mask, 0], pca_clap[mask, 1], alpha=0.3, s=8, label=ds)
    axes[1, 0].set_title('MERT Embeddings PCA (by Dataset)')
    axes[1, 0].legend()
    axes[1, 1].set_title('CLAP Embeddings PCA (by Dataset)')
    axes[1, 1].legend()
    plt.tight_layout()
    save_plot('10c_embedding_pca')
    
    # MERT + CLAP combined t-SNE
    X_emb = np.hstack([X_mert_scaled, X_clap_scaled])
    n_tsne = min(3000, len(X_emb))
    idx_tsne = np.random.choice(len(X_emb), n_tsne, replace=False)
    tsne_emb = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_emb[idx_tsne])
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    df_tsne = df.iloc[idx_tsne]
    for label, color, name in [(True, 'red', 'AI'), (False, 'blue', 'Real')]:
        mask = df_tsne['is_ai'].values == label
        axes[0].scatter(tsne_emb[mask, 0], tsne_emb[mask, 1], c=color, alpha=0.4, s=10, label=name)
    axes[0].set_title('MERT+CLAP t-SNE (Real vs AI)')
    axes[0].legend()
    
    for ds in df_tsne['dataset'].unique():
        mask = df_tsne['dataset'].values == ds
        axes[1].scatter(tsne_emb[mask, 0], tsne_emb[mask, 1], alpha=0.4, s=10, label=ds)
    axes[1].set_title('MERT+CLAP t-SNE (by Dataset)')
    axes[1].legend()
    plt.tight_layout()
    save_plot('10c_embedding_tsne')
    
    print('Embedding analysis complete.')
else:
    print('Embeddings not computed. Run precompute_all.py without --skip_embeddings for this analysis.')

Embedding analysis complete.


In [88]:
# === 10d. Cross-dataset domain shift analysis ===
# How different are features across datasets? (Are we comparing apples to apples?)
from scipy.stats import wasserstein_distance

datasets = df['dataset'].unique()
feature_sets = {
    'Classical (MFCC mean)': X_classical[:, 0],  # First MFCC mean
    'Classical (Tempo)': X_classical[:, 430],  # Tempo
    'AI: PCI': X_ai_det[:, 0],
    'AI: HNR': X_ai_det[:, 7],
    'AI: Artifact Energy': X_ai_det[:, 21],
    'Fakeprint Energy': fp_energy,
}

print('Wasserstein distances between dataset distributions:')
for feat_name, feat_vals in feature_sets.items():
    print(f'\n  {feat_name}:')
    for i, ds1 in enumerate(datasets):
        for ds2 in datasets[i+1:]:
            v1 = feat_vals[(df['dataset'] == ds1).values]
            v2 = feat_vals[(df['dataset'] == ds2).values]
            v1 = v1[~np.isnan(v1) & ~np.isinf(v1)]
            v2 = v2[~np.isnan(v2) & ~np.isinf(v2)]
            if len(v1) > 0 and len(v2) > 0:
                wd = wasserstein_distance(v1, v2)
                print(f'    {ds1} vs {ds2}: {wd:.4f}')

# Distribution overlay for key features
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.ravel()
for i, (feat_name, feat_vals) in enumerate(feature_sets.items()):
    if i >= len(axes): break
    for ds in datasets:
        mask = (df['dataset'] == ds).values
        vals = feat_vals[mask]
        vals = vals[~np.isnan(vals) & ~np.isinf(vals)]
        if len(vals) > 10:
            sns.kdeplot(vals, label=ds, ax=axes[i], fill=True, alpha=0.3)
    axes[i].set_title(feat_name)
    axes[i].legend(fontsize=8)
plt.suptitle('Feature Distributions Across Datasets', fontsize=13)
plt.tight_layout()
save_plot('10d_cross_dataset_distributions')

print('\nCross-dataset domain shift analysis complete.')

Wasserstein distances between dataset distributions:

  Classical (MFCC mean):
    sonics vs fakemusiccaps: 6.3646
    sonics vs smp: 2.4892
    sonics vs echoes: 3.4134
    fakemusiccaps vs smp: 6.6076
    fakemusiccaps vs echoes: 4.8105
    smp vs echoes: 4.6606

  Classical (Tempo):
    sonics vs fakemusiccaps: 5.0049
    sonics vs smp: 9.7436
    sonics vs echoes: 3.4162
    fakemusiccaps vs smp: 7.6521
    fakemusiccaps vs echoes: 2.1350
    smp vs echoes: 6.7330

  AI: PCI:
    sonics vs fakemusiccaps: 0.0163
    sonics vs smp: 0.0059
    sonics vs echoes: 0.0051
    fakemusiccaps vs smp: 0.0166
    fakemusiccaps vs echoes: 0.0148
    smp vs echoes: 0.0102

  AI: HNR:
    sonics vs fakemusiccaps: 0.9711
    sonics vs smp: 1.4720
    sonics vs echoes: 0.3566
    fakemusiccaps vs smp: 2.1099
    fakemusiccaps vs echoes: 1.0044
    smp vs echoes: 1.2242

  AI: Artifact Energy:
    sonics vs fakemusiccaps: 0.1507
    sonics vs smp: 0.0256
    sonics vs echoes: 0.0241
    fakemusiccap

In [111]:
# === Final Summary ===
print('=' * 70)
print('FEATURE EDA SUMMARY')
print('=' * 70)
print(f'Total files analyzed: {len(df)}')
for ds in df['dataset'].unique():
    sub = df[df['dataset'] == ds]
    n_ai = sub['is_ai'].sum()
    n_real = len(sub) - n_ai
    print(f'  {ds}: {len(sub)} total ({n_ai} AI, {n_real} real)')

print(f'\nFeature dimensions:')
print(f'  Classical: {X_classical.shape[1]}d')
print(f'  AI Detection: {X_ai_det.shape[1]}d')
print(f'  Fakeprint: {X_fakeprint.shape[1]}d')
print(f'  MERT: {X_mert.shape[1]}d (computed: {has_mert})')
print(f'  CLAP: {X_clap.shape[1]}d (computed: {has_clap})')

print(f'\nAll figures saved to: {FIG_DIR.resolve()}/')
print(f'Total figures: {len(list(FIG_DIR.glob("*.png")))}')

FEATURE EDA SUMMARY
Total files analyzed: 10178
  sonics: 4015 total (2962 AI, 1053 real)
  fakemusiccaps: 2000 total (2000 AI, 0 real)
  smp: 290 total (0 AI, 290 real)
  echoes: 3873 total (3577 AI, 296 real)

Feature dimensions:
  Classical: 432d
  AI Detection: 22d
  Fakeprint: 897d
  MERT: 1024d (computed: True)
  CLAP: 512d (computed: True)

All figures saved to: /home/geoka/projects/audio-project/eda/feature_figs/
Total figures: 43
